In [ ]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [ ]:

openai = OpenAI()
OPENAI_MODEL = "gpt-5"

In [74]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '10',
  'version': '10.0.26200',
  'kernel': '10',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': ''},
 'package_managers': ['winget', 'choco'],
 'cpu': {'brand': '11th Gen Intel(R) Core(TM) i5-1135G7 @ 2.40GHz',
  'cores_logical': 8,
  'cores_physical': 4,
  'simd': []},
 'toolchain': {'compilers': {'gcc': '', 'g++': '', 'clang': '', 'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [75]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openai.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

You don’t currently have a C/C++ compiler installed. Your system report shows gcc, g++, clang, and MSVC (cl) are all missing.

Simplest way to install a compiler on your Windows 10 system (using Chocolatey):
- Open PowerShell as Administrator.
- Run: choco install mingw -y
- Close and reopen your terminal (or run refreshenv), then verify: g++ --version
- If g++ isn’t found, add C:\tools\mingw64\bin to your PATH and try again.

Once installed, use these exact commands in Python for fastest typical runtime performance:
- Note: -O3 -march=native -flto -DNDEBUG aim for high performance. If the link step fails with -flto, remove -flto and try again.

compile_command = ["g++", "-std=c++20", "-O3", "-march=native", "-flto", "-DNDEBUG", "main.cpp", "-o", "main.exe"]
run_command = ["main.exe"]

Alternative (winget, if you prefer):
- winget install MSYS2.MSYS2
- Launch the “MSYS2 UCRT64” shell and run: pacman -S --needed mingw-w64-ucrt-x86_64-gcc
- Then, from that same UCRT64 environment, you can compile with the same g++ options as above. If calling from Python in a normal Windows shell, use the full path to g++, e.g., C:\msys64\ucrt64\bin\g++.exe.

In [76]:
compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

In [77]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
The execution command is:
{run_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [78]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]

In [79]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [80]:
def port(client, model, python):
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)

In [81]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [82]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [83]:
run_python(pi)

Result: 3.141592656089
Execution Time: 27.217992 seconds


In [84]:
port(openai, OPENAI_MODEL, pi)

In [85]:
# Use the commands from GPT 5

def compile_and_run():
    subprocess.run(compile_command, check=True, text=True, capture_output=True)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)

In [86]:
compile_and_run()

FileNotFoundError: [WinError 2] The system cannot find the file specified